# Genie Agents Workshop: Ask Northgate's Billing Data in Plain English

Attendees build a Genie Agent over the Northgate Advisory LLP consulting data from the Lakeflow Designer workshop. They curate tables with good metadata, add instructions and example SQL, and test with benchmarks. [3. Agents](#notebook-3452153026155249) reuses this agent as a tool in an AI Playground agent.

## What is an agent?

An agent isn't a single model. It's a small system made of four parts:

| Part | What it is | In this notebook |
| --- | --- | --- |
| **Governed enterprise data** | The facts the agent is allowed to use, protected by Unity Catalog permissions | The tables behind your Genie Agent: contract terms from the signed SOWs, agreed and standard day rates, engagements |
| **An LLM** | The model that reads, reasons and writes | Any tools-enabled model in Playground, and you can swap it |
| **Logic** | The rules: instructions and code | The system prompt, plus the instructions and example SQL inside the Genie Agent |
| **Tool calls** | Actions the model can take, including calling other agents | The Genie Agent you built in Notebook 1, used as a tool |

The model decides **when** to call a tool and **what to ask** it. The Genie Agent decides **what** data it can see, with the user's own permissions.

**The scenario.** An engagement manager types: *"I have a new client I need to draft an SOW for, can you create me the same type of SOW?"* The agent asks the Genie Agent for the terms of a similar signed SOW, prices the team at today's standard rates and writes a draft for the partner to review.

| Part | Topic | Format |
| --- | --- | --- |
| 1 | Build the agent in AI Playground | Hands-on |
| 2 | Swap models and compare the drafts | Hands-on |
| 3 | Turn the agent into an app | Hands-on |

**Prerequisites:** you've built the Genie Agent in [1. Genie Agents](#notebook-3452153026155247) and added the contract tables (`contract_terms`, `contracted_rates`) to it in Part 1B of [2. Document Intelligence](#notebook-3452153026155248). Part 3 needs Databricks Apps and the **Managed MCP Servers** preview enabled in the workspace.

## What you will build

The Northgate finance team asks the same questions every month: *How much WIP has each engagement built up? Who is under their utilisation target? Which clients have the most rechargeable expenses?* Today an analyst writes SQL for each one. A **Genie Agent** lets the partners and managers ask these questions themselves, in plain English, and get back the SQL, a table and a chart.

A Genie Agent is only as good as the context it's given. Most of this tutorial is about **curating** that context: clean tables, clear descriptions, business definitions, example queries and tests.

| Part | Topic | Format |
| --- | --- | --- |
| 0 | Prerequisites and configuration | Run cells |
| 1 | Prepare Genie-ready tables (clean data, table and column descriptions) | Run cells, walkthrough |
| 2 | Create the Genie Agent | Hands-on |
| 3 | Teach it the business: instructions, joins, SQL expressions, example SQL | Hands-on |
| 4 | Ask questions: simple, ambiguous, follow-ups and charts | Hands-on |
| 5 | Agent mode: multi-step questions | Hands-on |
| 6 | Benchmarks: measure accuracy before you share | Hands-on |
| 7 | Share it safely | Walkthrough |

**Prerequisite:** the facilitator has run the [1. Workshop Data Setup](#notebook-3120290690157533) notebook from the Lakeflow Designer workshop, so the `consulting_billing` schema contains the `employees`, `clients`, `engagements` and `engagement_assignments` tables and the raw files volume.

## Part 0: Configuration

Set the widgets to the catalog, schema and volume used by the setup notebook. The facilitator runs Parts 0 and 1 once; attendees share the resulting tables and each build their own Genie Agent on top of them.

In [0]:
dbutils.widgets.text("catalog", "praju_stable_2_catalog", "Catalog")
dbutils.widgets.text("schema", "consulting_billing", "Schema")
dbutils.widgets.text("volume", "raw_files", "Volume")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")

FQ = f"`{CATALOG}`.`{SCHEMA}`"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
spark.sql(f"USE {FQ}")

missing = [t for t in ["employees", "clients", "engagements", "engagement_assignments"]
           if not spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{t}")]
if missing:
    raise RuntimeError(f"Missing tables {missing}: run the '1. Workshop Data Setup' notebook first.")

print(f"Schema : {CATALOG}.{SCHEMA}")
print(f"Volume : {VOLUME_PATH}")

## Part 1: Prepare Genie-ready tables

Genie writes SQL against the tables you give it, so it inherits every problem in those tables. The raw timesheet CSVs have text dates, padded engagement codes and `y`/`Y` flags. If you point Genie at those, it has to clean the data in every query, and it will sometimes get it wrong.

**Rule of thumb: give Genie the same clean, modelled tables you would give a new analyst.** The next cell builds three:

| Table | Grain | Built from |
| --- | --- | --- |
| `timesheet_entries` | One row per timesheet entry | `raw_files/timesheets/*.csv`, cleaned (the same logic as the Lakeflow Designer flow) |
| `grade_rate_card` | One row per grade | `raw_files/reference/grade_rate_card.csv` |
| `expense_claims` | One row per expense claim, with the amount converted to GBP | `raw_files/expenses/*.json` + `fx_rates_monthly.csv` |

The cell after that adds **table and column descriptions** in Unity Catalog. Genie reads these to decide which table and column a question refers to, so they matter as much as the data. Good descriptions say what a row is, what the codes mean and which values to filter on.

In [0]:
# Clean timesheets: the same rules as the Lakeflow Designer flow
spark.sql(f"""
CREATE OR REPLACE TABLE timesheet_entries AS
SELECT
  timesheet_id,
  employee_id,
  upper(trim(engagement_code))                                AS engagement_code,
  to_date(work_date, 'dd/MM/yyyy')                            AS work_date,
  CAST(date_trunc('MONTH', to_date(work_date, 'dd/MM/yyyy')) AS DATE) AS billing_month,
  CAST(hours AS DOUBLE)                                       AS hours,
  ROUND(CAST(hours AS DOUBLE) / 7.5, 4)                       AS days,
  activity,
  upper(trim(billable)) = 'Y'                                 AS is_billable,
  approval_status,
  dayofweek(to_date(work_date, 'dd/MM/yyyy')) IN (1, 7)       AS is_weekend,
  comments
FROM read_files('{VOLUME_PATH}/timesheets/', format => 'csv', header => true, inferSchema => false)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE grade_rate_card AS
SELECT grade,
       CAST(grade_level AS INT)              AS grade_level,
       CAST(standard_day_rate_gbp AS DOUBLE) AS standard_day_rate_gbp,
       CAST(cost_day_rate_gbp AS DOUBLE)     AS cost_day_rate_gbp,
       CAST(effective_from AS DATE)          AS effective_from
FROM read_files('{VOLUME_PATH}/reference/grade_rate_card.csv', format => 'csv', header => true, inferSchema => false)
""")

# Expenses: convert to GBP with the FX rate for the month of the expense
spark.sql(f"""
CREATE OR REPLACE TABLE expense_claims AS
WITH fx AS (
  SELECT month, currency, CAST(rate_to_gbp AS DOUBLE) AS rate_to_gbp
  FROM read_files('{VOLUME_PATH}/reference/fx_rates_monthly.csv', format => 'csv', header => true, inferSchema => false)
)
SELECT
  x.expense_id,
  x.employee_id,
  x.engagement_code,
  CAST(x.expense_date AS DATE)                                AS expense_date,
  CAST(date_trunc('MONTH', CAST(x.expense_date AS DATE)) AS DATE) AS expense_month,
  x.category,
  x.merchant,
  x.description,
  CAST(x.amount AS DOUBLE)                                    AS amount,
  x.currency,
  fx.rate_to_gbp,
  ROUND(CAST(x.amount AS DOUBLE) * fx.rate_to_gbp, 2)         AS amount_gbp,
  CAST(x.rechargeable AS BOOLEAN)                             AS is_rechargeable,
  CAST(x.receipt_attached AS BOOLEAN)                         AS receipt_attached,
  x.status,
  CAST(x.submitted_at AS TIMESTAMP)                           AS submitted_at
FROM read_files('{VOLUME_PATH}/expenses/', format => 'json') x
LEFT JOIN fx ON fx.month = substr(x.expense_date, 1, 7) AND fx.currency = x.currency
""")

display(spark.sql("""
  SELECT 'timesheet_entries' AS table_name, COUNT(*) AS row_count FROM timesheet_entries
  UNION ALL SELECT 'grade_rate_card', COUNT(*) FROM grade_rate_card
  UNION ALL SELECT 'expense_claims', COUNT(*) FROM expense_claims
  UNION ALL SELECT 'expense_claims without an FX rate', COUNT(*) FROM expense_claims WHERE amount_gbp IS NULL
"""))

In [0]:
# Table and column descriptions in Unity Catalog. Genie reads these to understand the data.
DOCS = {
    "employees": ("One row per Northgate Advisory employee (60 staff). Use full_name when showing people.", {
        "employee_id": "Unique employee identifier, e.g. E0001",
        "full_name": "Employee first and last name",
        "grade": "Job grade, from most to least senior: Partner, Director, Senior Manager, Manager, Assistant Manager, Senior Consultant, Consultant, Analyst",
        "grade_level": "Numeric seniority of the grade: 8 = Partner down to 1 = Analyst",
        "practice": "The practice the employee belongs to: Audit, Technology Consulting, Deal Advisory, Risk Consulting or Tax",
        "office": "UK office the employee is based in",
        "manager_id": "employee_id of the employee's line manager (self-join to employees). Null for the most senior staff",
        "annual_utilisation_target": "Target share of available hours that should be billable, as a decimal (0.85 = 85%)",
        "is_active": "True if the employee currently works at the firm",
    }),
    "clients": ("One row per client of the firm (12 clients).", {
        "client_id": "Unique client identifier",
        "client_name": "Client legal name",
        "industry": "Client industry sector",
        "billing_currency": "Currency the client is invoiced in: GBP, EUR or USD",
        "client_tier": "Client importance tier",
        "relationship_partner_id": "employee_id of the partner who owns the client relationship",
    }),
    "engagements": ("One row per engagement (job code). engagement_type = 'Client' for the 16 chargeable client engagements; 'Internal' for the 4 internal time codes (INT-LVE leave, INT-TRN training, INT-BD business development, INT-ADM admin).", {
        "engagement_code": "Unique engagement code, e.g. ENG-26-003. Internal codes start with INT-",
        "engagement_name": "Short name of the engagement",
        "client_id": "Client the engagement is for. Null for internal codes",
        "practice": "Practice that runs the engagement",
        "engagement_type": "Client (chargeable) or Internal (non-chargeable time code)",
        "billing_type": "Commercial basis: Time & Materials, Capped T&M or Fixed Fee",
        "engagement_partner_id": "employee_id of the engagement partner",
        "engagement_manager_id": "employee_id of the engagement manager",
        "sow_reference": "Reference of the signed Statement of Work PDF, e.g. SOW-26-003",
    }),
    "engagement_assignments": ("Who is staffed on which engagement, with their role and planned allocation.", {
        "allocation_pct": "Planned share of the person's time on this engagement, as a decimal (0.5 = 50%)",
        "assigned_from": "First day of the assignment",
        "assigned_to": "Last day of the assignment",
    }),
    "timesheet_entries": ("One row per timesheet entry: hours an employee booked to an engagement on a day, Aug-Nov 2026. Only approved, billable entries on client engagements count towards WIP and utilisation.", {
        "engagement_code": "Engagement the time was booked to (join to engagements)",
        "work_date": "Date the work was done",
        "billing_month": "First day of the month of work_date. Use for monthly reporting",
        "hours": "Hours booked. A working day is 7.5 hours",
        "days": "hours / 7.5. Multiply by a day rate to get a value in GBP",
        "is_billable": "True if the time is chargeable to the client",
        "approval_status": "Approved, Submitted (awaiting approval) or Rejected. Only Approved time is final",
        "is_weekend": "True if work_date is a Saturday or Sunday",
    }),
    "grade_rate_card": ("Standard (list) day rate charged to clients and internal cost day rate for each grade, in GBP.", {
        "standard_day_rate_gbp": "List price per day for the grade before any client discount",
        "cost_day_rate_gbp": "Internal cost per day for the grade (salary and overheads)",
    }),
    "expense_claims": ("One row per expense claim made by an employee against an engagement, with the amount converted to GBP.", {
        "expense_month": "First day of the month of expense_date",
        "category": "Rail travel, Hotel, Subsistence, Taxi or Mileage",
        "amount": "Claim amount in the original currency",
        "currency": "Currency of amount",
        "amount_gbp": "Claim amount converted to GBP at the monthly FX rate. Use this for totals",
        "is_rechargeable": "True if the expense can be recharged to the client",
        "status": "Approved or Pending",
    }),
}

for table, (table_comment, columns) in DOCS.items():
    spark.sql(f"COMMENT ON TABLE {table} IS '{table_comment.replace(chr(39), chr(39) * 2)}'")
    for col, comment in columns.items():
        spark.sql(f"ALTER TABLE {table} ALTER COLUMN {col} COMMENT '{comment.replace(chr(39), chr(39) * 2)}'")
    print(f"{table}: table comment + {len(columns)} column comments")

## Part 2: Create the Genie Agent (hands-on)

1. In the left navigation click **Genie**, then **+ New**.
2. Add these tables from `<catalog>.consulting_billing`:
   * `timesheet_entries`, `employees`, `engagements`, `clients`, `engagement_assignments`, `grade_rate_card`, `expense_claims`
   * Do **not** add the `nb_*` tables or the raw files. Fewer, cleaner tables give better answers.
3. Open **Settings**:
   * **Title:** `<initials> Northgate Billing & Utilisation`
   * **Description:** *Ask questions about timesheets, WIP, utilisation, rates and expenses for Northgate Advisory's client engagements (Aug-Oct 2026).*
   * **Default warehouse:** a serverless SQL warehouse.
4. Add three **sample questions**. These are shown to users as starting points:
   * *What was the WIP at standard rates by engagement in October 2026?*
   * *Which employees were below their utilisation target in September 2026?*
   * *Show rechargeable expenses in GBP by client.*
5. Ask your first question: *How many billable hours were booked in October 2026?* Click **Show code** to read the SQL Genie wrote. Did it filter on `approval_status`? Did it exclude internal codes? (Without instructions it usually doesn't. You fix that in Part 3.)

## Part 3: Teach it the business (hands-on)

Genie knows SQL. It doesn't know what Northgate means by *WIP* or *utilisation*. Add that knowledge in the **Configure** section of the agent, in this order (most to least effective):

### 3a. Text instructions
Paste this into **Instructions > Text**:

```
You are answering questions for the finance team of Northgate Advisory LLP, a UK consulting firm.
- "WIP" (work in progress) = days x day rate for approved, billable time on client engagements:
  timesheet_entries.approval_status = 'Approved' AND timesheet_entries.is_billable AND engagements.engagement_type = 'Client'.
  "WIP at standard rates" uses grade_rate_card.standard_day_rate_gbp for the employee's grade.
- "Cost" = days x grade_rate_card.cost_day_rate_gbp. "Margin" = WIP - cost.
- "Billable hours" always means approved, billable hours on client engagements unless the user says otherwise.
- "Utilisation" = billable hours / available hours. Available hours = working days in the month x 7.5
  (Monday-Friday, excluding UK bank holidays; 31 August 2026 is a bank holiday).
- "Practice" means the engagement's practice (engagements.practice) for questions about work or WIP,
  and the employee's practice (employees.practice) for questions about people or headcount.
- Show people by full_name and engagements by engagement_code and engagement_name.
- Money is in GBP. Round money to whole pounds and percentages to one decimal place.
- Data covers August to October 2026 (plus 2-6 November 2026 after the November file is loaded).
  If the user asks about a period with no data, say so instead of returning zero.
```

### 3b. Joins
Genie infers most joins from column names, but say them explicitly under **Joins** so it never guesses:

| Left | Right | Condition | Relationship |
| --- | --- | --- | --- |
| `timesheet_entries` | `employees` | `employee_id = employee_id` | Many to one |
| `timesheet_entries` | `engagements` | `engagement_code = engagement_code` | Many to one |
| `employees` | `grade_rate_card` | `grade = grade` | Many to one |
| `engagements` | `clients` | `client_id = client_id` | Many to one |
| `expense_claims` | `engagements` | `engagement_code = engagement_code` | Many to one |
| `engagement_assignments` | `employees` | `employee_id = employee_id` | Many to one |

### 3c. SQL expressions (measures, filters, dimensions)
These give a name to a business concept so users can say it naturally:

| Type | Name | SQL | Synonyms |
| --- | --- | --- | --- |
| Measure | WIP at standard rates | `SUM(timesheet_entries.days * grade_rate_card.standard_day_rate_gbp)` | WIP, work in progress |
| Measure | Cost | `SUM(timesheet_entries.days * grade_rate_card.cost_day_rate_gbp)` | staff cost |
| Filter | Chargeable time | `timesheet_entries.approval_status = 'Approved' AND timesheet_entries.is_billable` | billable time, chargeable hours |
| Filter | Weekend working | `timesheet_entries.is_weekend` | weekend entries |
| Dimension | Billing month | `timesheet_entries.billing_month` | month |

### 3d. Example SQL queries
Add these under **SQL queries**. Use the most typical phrasing of the question as the title.

**What was the WIP at standard rates by engagement in October 2026?**
```sql
SELECT e.engagement_code, e.engagement_name, c.client_name,
       ROUND(SUM(t.days * r.standard_day_rate_gbp)) AS wip_standard_gbp
FROM timesheet_entries t
JOIN employees emp ON t.employee_id = emp.employee_id
JOIN grade_rate_card r ON emp.grade = r.grade
JOIN engagements e ON t.engagement_code = e.engagement_code
JOIN clients c ON e.client_id = c.client_id
WHERE t.approval_status = 'Approved' AND t.is_billable AND e.engagement_type = 'Client'
  AND t.billing_month = DATE'2026-10-01'
GROUP BY ALL
ORDER BY wip_standard_gbp DESC
```

**Which timesheet entries look unusual?** (weekend, over 12 hours, or not assigned)
```sql
SELECT t.timesheet_id, emp.full_name, t.engagement_code, t.work_date, t.hours,
       t.is_weekend, t.hours > 12 AS over_12_hours, a.assignment_id IS NULL AS not_assigned
FROM timesheet_entries t
JOIN employees emp ON t.employee_id = emp.employee_id
JOIN engagements e ON t.engagement_code = e.engagement_code
LEFT JOIN engagement_assignments a
  ON a.employee_id = t.employee_id AND a.engagement_code = t.engagement_code
WHERE e.engagement_type = 'Client'
  AND (t.is_weekend OR t.hours > 12 OR a.assignment_id IS NULL)
ORDER BY t.work_date
```

## Part 4: Ask questions (hands-on)

Work through these in order. For every answer, click **Show code** and check the SQL before you trust the number. Expected answers are in the answer key at the end of this notebook.

| # | Type | Question | What to look for |
| --- | --- | --- | --- |
| 1 | Simple | *How many billable hours were booked in October 2026?* | Filters on Approved, billable and Client (from your instructions) |
| 2 | Join + measure | *What was the WIP at standard rates by engagement in October 2026?* | Uses your example SQL |
| 3 | Ranking | *Which 5 employees had the most billable hours in September 2026?* | Shows full_name, not employee_id |
| 4 | Business definition | *Who was below their utilisation target in October 2026?* | Uses your utilisation instruction: 22 working days x 7.5 = 165 available hours in October |
| 5 | Ambiguous | *What is the WIP by practice?* | Uses engagement practice (your instruction); ask *"and by the employee's practice?"* to see the difference |
| 6 | Self-join | *Who is the line manager of each Manager in the Audit practice?* | Self-join on manager_id |
| 7 | Anomalies | *How many timesheet entries were on a weekend, and who made them?* | 12 entries |
| 8 | Follow-up | Ask *"Show rechargeable expenses in GBP by client"*, then *"just for Rail travel"*, then *"as a bar chart"* | Keeps the context between questions |
| 9 | No data | *What was the WIP in March 2026?* | Says there is no data rather than returning 0 |

**When Genie gets it wrong:** don't just retry. Fix the cause: edit the SQL and click **Save as instruction** or add it as an example query, add a column description, or clarify a text instruction. Then ask again.

**Thumbs up/down:** users' feedback appears in the **Monitoring** tab, so you can see which questions to improve next.

## Part 5: Agent mode (hands-on)

Chat mode answers one question with one SQL query. **Agent mode** plans several steps, runs multiple queries, and writes a short report with its reasoning. Switch to Agent mode in the chat box and try:

* *Compare the margin at standard rates across the Technology Consulting engagements and explain the differences using grade mix and hours.*
* *Prepare a utilisation review for October 2026: who is below target, which practices have spare capacity, and which engagements could use them.*
* *Investigate the unusual timesheet entries. Are they concentrated in particular people, engagements or weeks?*

Compare the answer with Chat mode. Agent mode is slower but better for open questions. Chat mode is better for a precise number. In [2. Document Intelligence](#notebook-3452153026155248) you'll use Agent mode to read the Statement of Work PDFs as well.

## Part 6: Benchmarks (hands-on)

Before you share the agent, prove it answers the important questions correctly. **Benchmarks** are a test suite: questions plus the SQL that gives the right answer. Genie runs each question as a new conversation and compares its result with yours.

1. Open the **Benchmarks** tab and click **Add benchmark**.
2. Add at least these four. For each, paste the question and the ground-truth SQL:

| Question | Ground-truth SQL |
| --- | --- |
| *How many billable hours were booked in October 2026?* | `SELECT SUM(t.hours) FROM timesheet_entries t JOIN engagements e USING (engagement_code) WHERE t.approval_status = 'Approved' AND t.is_billable AND e.engagement_type = 'Client' AND t.billing_month = DATE'2026-10-01'` |
| *How many timesheet entries were on a weekend?* | `SELECT COUNT(*) FROM timesheet_entries WHERE is_weekend` |
| *What was the total WIP at standard rates in October 2026?* | `SELECT ROUND(SUM(t.days * r.standard_day_rate_gbp)) FROM timesheet_entries t JOIN employees emp USING (employee_id) JOIN grade_rate_card r ON emp.grade = r.grade JOIN engagements e USING (engagement_code) WHERE t.approval_status = 'Approved' AND t.is_billable AND e.engagement_type = 'Client' AND t.billing_month = DATE'2026-10-01'` |
| *What is the total rechargeable expense in GBP by client?* | `SELECT c.client_name, ROUND(SUM(x.amount_gbp)) AS rechargeable_gbp FROM expense_claims x JOIN engagements e USING (engagement_code) JOIN clients c USING (client_id) WHERE x.is_rechargeable GROUP BY c.client_name` |

3. Add two **phrasings** of the same question (for example *"chargeable hours last October"*). Users rarely use your exact wording.
4. Click **Run all benchmarks** (Chat mode). Review any failures, fix the instructions, and run again. Record the score, and re-run the benchmarks after every change to the instructions; this is the evidence you'd show a reviewer that the agent is fit to share.

**Discussion (Audit/Advisory):** a benchmark suite is the Genie equivalent of testing a control. It's repeatable, documented, and re-runnable after every change to the agent or the data.

## Part 7: Share it safely

* **Permissions:** share the agent with **Can Run** for users and **Can Edit** for co-authors. Users also need `SELECT` on the tables. Genie always runs with the asking user's own permissions, so row filters and column masks still apply.
* **Where users find it:** the Genie page, **Genie One** (which can route business questions to your agent), an embedded iframe in an internal app (Share > Embed space, after an admin allows it), or as a tool for other agents, like the AI Playground agent in [3. Agents](#notebook-3452153026155249).
* **Keep it accurate:** review the **Monitoring** tab weekly. Turn thumbs-down answers into new instructions, examples or benchmarks, and re-run the benchmarks after every change.

**Next:** [2. Document Intelligence](#notebook-3452153026155248) adds the Statement of Work PDFs to this agent.

## Facilitator answer key (Aug-Oct 2026 data, before the November file is added)

| # | Question | Expected answer |
| --- | --- | --- |
| 1 | Billable hours, October 2026 | 4,152 hours |
| 2 | Top engagement by WIP at standard rates, October 2026 | ENG-26-001 FY26 Statutory Audit, about GBP 57,910 |
| 3 | Most billable hours, September 2026 | Evie Roberts, 161 hours |
| 4 | Employees below utilisation target, October 2026 | 53 of 60 (only 7 people are at or above target) |
| 7 | Weekend timesheet entries | 12 |
| 8 | Top client by rechargeable expenses | Harbourline Bank plc, about GBP 2,397 |
| 9 | Timesheet entries in March 2026 | 0 rows: Genie should say there's no data |
| Benchmark | Total WIP at standard rates, October 2026 | GBP 533,519 |

53 of 60 below target looks alarming, and it's a good discussion point rather than a data error. Utilisation is measured monthly against an annual target. Several staff are on the bench in October because their assignments ended, and partners spend much of their time on business development. Ask attendees to have Genie explain *why* each group is below target.

The WIP figure can differ by GBP 1 from the Lakeflow Designer answer-key table, because that table rounds each month before summing.